# Student Depression Detector — Model Training

This notebook trains two machine learning models to predict student depression:
- **Logistic Regression**
- **Random Forest Classifier**

The best model is automatically selected by test accuracy and saved to `model/` for use in the GUI.

**Pipeline:**
```
Load Data → Preprocessing → Feature Engineering → Scale → Split → Train → Evaluate → Save
```

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Imports OK')

## 2. Paths

In [ ]:
# Resolve paths relative to this notebook's location
NOTEBOOK_DIR = os.path.abspath('')          # directory of this notebook
BASE_DIR     = os.path.dirname(NOTEBOOK_DIR) # project root
DATA_PATH    = os.path.join(BASE_DIR, 'Student Depression Dataset.csv')
MODEL_DIR    = os.path.join(BASE_DIR, 'model')

os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Data  : {DATA_PATH}')
print(f'Model : {MODEL_DIR}')

## 3. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 4. Preprocessing

Steps:
1. Drop duplicates
2. Fill missing values (median for numeric, mode for categorical)
3. Encode target `Depression` as 0/1
4. LabelEncode all other categorical columns

In [ ]:
# --- 4a. Drop duplicates ---
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Dropped {before - len(df)} duplicate rows. Remaining: {len(df)}')

In [ ]:
# --- 4b. Fill missing values ---
numeric_cols     = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f'  Filled numeric "{col}" with median')

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])
        print(f'  Filled categorical "{col}" with mode')

print(f'Missing values after fill: {df.isnull().sum().sum()}')

In [ ]:
# --- 4c. Encode target variable ---
TARGET = 'Depression'

if df[TARGET].dtype == object:
    df[TARGET] = df[TARGET].map({'Yes': 1, 'No': 0})
    print(f'Encoded "{TARGET}": Yes->1, No->0')
else:
    print(f'"{TARGET}" is already numeric (0/1)')

print(f'Class distribution:\n{df[TARGET].value_counts()}')

In [ ]:
# --- 4d. LabelEncode all other categorical columns ---
encoder_dict = {}
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoder_dict[col] = le
    print(f'  LabelEncoded "{col}" -- {len(le.classes_)} classes')

print(f'\nTotal encoded columns: {len(encoder_dict)}')

## 5. Feature Engineering

| New Feature | Description |
|---|---|
| `Stress_Score` | Mean of Academic Pressure + Work Pressure + Financial Stress |
| `Sleep_Category` | 0=Low (<6h), 1=Normal (6-8h), 2=High (>8h) |
| `Study_Work_Balance` | Study Hours / (Study Hours + 1) |

In [ ]:
# --- 5a. Stress_Score ---
stress_cols = [c for c in ['Academic Pressure', 'Work Pressure', 'Financial Stress'] if c in df.columns]
if stress_cols:
    df['Stress_Score'] = df[stress_cols].mean(axis=1)
    print(f'Created Stress_Score from: {stress_cols}')

# --- 5b. Sleep_Category ---
SLEEP_COL = 'Sleep Duration'
if SLEEP_COL in df.columns and SLEEP_COL in encoder_dict:
    sleep_le = encoder_dict[SLEEP_COL]
    sleep_hour_map = {}
    for i, cls in enumerate(sleep_le.classes_):
        cls_low = cls.lower()
        if 'less than 5' in cls_low:  sleep_hour_map[i] = 4.0
        elif '5-6' in cls_low:        sleep_hour_map[i] = 5.5
        elif '7-8' in cls_low:        sleep_hour_map[i] = 7.5
        elif 'more than 8' in cls_low: sleep_hour_map[i] = 9.0
        else:                          sleep_hour_map[i] = 7.0

    df['Sleep_Hours_Approx'] = df[SLEEP_COL].map(sleep_hour_map).fillna(7.0)
    df['Sleep_Category'] = df['Sleep_Hours_Approx'].apply(
        lambda h: 0 if h < 6 else (1 if h <= 8 else 2)
    )
    df.drop(columns=['Sleep_Hours_Approx'], inplace=True)
    print('Created Sleep_Category (0=Low, 1=Normal, 2=High)')

# --- 5c. Study_Work_Balance ---
study_col = 'Work/Study Hours'
if study_col in df.columns:
    df['Study_Work_Balance'] = df[study_col] / (df[study_col] + 1)
    print('Created Study_Work_Balance')

print(f'\nShape after feature engineering: {df.shape}')

## 6. Scale Features & Train/Test Split

In [ ]:
# Separate features (X) and target (y)
X = df.drop(columns=[TARGET])
y = df[TARGET]

feature_columns = X.columns.tolist()
print(f'Features ({len(feature_columns)}): {feature_columns}')

In [ ]:
# Scale using StandardScaler
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler for the GUI
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
print('Scaler saved -> model/scaler.pkl')

In [ ]:
# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42
)
print(f'Train: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows')

## 7. Train & Evaluate Models

We train **Logistic Regression** and **Random Forest** then compare their test accuracy.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
}

results = {}

for model_name, model in models.items():
    print('=' * 55)
    print(f'  {model_name}')
    print('=' * 55)

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test,  y_test_pred)
    results[model_name] = {'model': model, 'test_acc': test_acc, 'y_pred': y_test_pred}

    print(f'  Train Accuracy : {train_acc*100:.2f}%')
    print(f'  Test  Accuracy : {test_acc*100:.2f}%')
    print()
    print(classification_report(y_test, y_test_pred, target_names=['Not Depressed','Depressed']))

In [ ]:
# Plot confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (model_name, info) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, info['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Not Depressed','Depressed'],
        yticklabels=['Not Depressed','Depressed'],
        ax=ax
    )
    ax.set_title(f'Confusion Matrix\n{model_name} (acc={info["test_acc"]*100:.1f}%)')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

    # Save individual PNG
    safe = model_name.replace(' ','_').lower()
    fig_single, ax_single = plt.subplots(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Not Depressed','Depressed'],
                yticklabels=['Not Depressed','Depressed'], ax=ax_single)
    ax_single.set_title(f'Confusion Matrix -- {model_name}')
    ax_single.set_xlabel('Predicted'); ax_single.set_ylabel('Actual')
    fig_single.tight_layout()
    fig_single.savefig(os.path.join(MODEL_DIR, f'confusion_matrix_{safe}.png'), dpi=150)
    plt.close(fig_single)

fig.tight_layout()
plt.show()

## 8. Model Selection & Save Artifacts

In [ ]:
# Select best model by test accuracy
best_name  = max(results, key=lambda k: results[k]['test_acc'])
best_model = results[best_name]['model']
best_acc   = results[best_name]['test_acc']

print(f'Best Model   : {best_name}')
print(f'Test Accuracy: {best_acc*100:.2f}%')
print(f'Reason       : Highest test accuracy among all trained models')

In [ ]:
# Save model, encoders, feature column list
joblib.dump(best_model,      os.path.join(MODEL_DIR, 'depression_model.pkl'))
joblib.dump(encoder_dict,    os.path.join(MODEL_DIR, 'encoders.pkl'))
joblib.dump(feature_columns, os.path.join(MODEL_DIR, 'feature_columns.pkl'))

for fname in ['depression_model.pkl', 'encoders.pkl', 'feature_columns.pkl', 'scaler.pkl']:
    path = os.path.join(MODEL_DIR, fname)
    size = os.path.getsize(path)
    print(f'  {fname:<30} {size:>8,} bytes  -- saved')

## 9. Feature Importance / Coefficients

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_path = os.path.join(MODEL_DIR, 'feature_importance.png')

if hasattr(best_model, 'feature_importances_'):
    # Random Forest
    fi_df = pd.DataFrame({'Feature': feature_columns, 'Importance': best_model.feature_importances_})
    fi_df.sort_values('Importance', ascending=True, inplace=True)
    fi_df.tail(20).plot.barh(x='Feature', y='Importance', ax=ax, color='steelblue', legend=False)
    ax.set_title(f'Feature Importances -- {best_name}')
    ax.set_xlabel('Importance Score')
else:
    # Logistic Regression
    coef_df = pd.DataFrame({'Feature': feature_columns, 'Coefficient': best_model.coef_[0]})
    coef_df.sort_values('Coefficient', ascending=True, inplace=True)
    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']]
    ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Logistic Regression Coefficients -- {best_name}')
    ax.set_xlabel('Coefficient Value')

plt.tight_layout()
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Chart saved -> {plot_path}')

## 10. Column Summary Table

In [ ]:
summary_rows = []
for col in df.columns:
    summary_rows.append({
        'Column':  col,
        'dtype':   str(df[col].dtype),
        'Role':    'Target' if col == TARGET else ('Engineered' if col in ['Stress_Score','Sleep_Category','Study_Work_Balance'] else 'Feature'),
        'Encoded': 'Yes' if col in encoder_dict else 'No',
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 11. Training Results Summary

In [ ]:
# Final accuracy comparison
acc_df = pd.DataFrame([
    {'Model': name, 'Test Accuracy (%)': round(info['test_acc']*100, 2)}
    for name, info in results.items()
]).sort_values('Test Accuracy (%)', ascending=False).reset_index(drop=True)

acc_df['Selected'] = acc_df['Model'].apply(lambda m: 'YES' if m == best_name else '')
print(acc_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#2563EB' if m == best_name else '#94A3B8' for m in acc_df['Model']]
bars = ax.bar(acc_df['Model'], acc_df['Test Accuracy (%)'], color=colors)
ax.set_ylim(75, 100)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Model Comparison -- Test Accuracy')
for bar, val in zip(bars, acc_df['Test Accuracy (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Training complete! All artifacts saved to model/')